# Notebook 09 — StratLake Strategy Comparison and Research Review

This standalone notebook restores or attaches to the StratLake feature/artifact/config archive produced by the previous notebooks, verifies the native configuration surface, runs available native StratLake strategies over the Q1 analysis window, parses native CLI output into comparison rows, discovers native artifacts, plots comparison summaries, and prepares an optional archive checkpoint.

Design boundary:

- Native Fintech and StratLake commands are preferred.
- Notebook code is used for orchestration, parsing, display, and artifact review.
- The notebook does not implement custom strategy logic or replace StratLake-native normalization, backtesting, feature generation, or archive behavior.

## 1. Install notebook dependencies and app packages

In [ ]:
!pip install "pandas-market-calendars>=5.0"
!pip install -i https://test.pypi.org/simple/ fintech-market-ingestion
!pip install -i https://test.pypi.org/simple/ stratlake-trade-engine

## 2. Imports, Colab detection, and Google Drive auth

In [ ]:
from pathlib import Path
import os
import re
import sys
import json
import subprocess
import getpass
from datetime import datetime, timezone

import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    drive = None
    IN_COLAB = False

print("IN_COLAB:", IN_COLAB)
print("Python executable:", sys.executable)
print("Current working directory:", Path.cwd().as_posix())

if IN_COLAB:
    drive.mount("/content/drive")
    print("Google Drive mounted.")
else:
    print("Not running in Colab; skipping Google Drive mount.")

## 3. Load Alpaca environment variables

In [ ]:
try:
    from google.colab import userdata
except Exception:
    userdata = None

def get_secret_or_prompt(name: str) -> str:
    value = None
    if userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
    if not value:
        value = getpass.getpass(f"Enter {name}: ")
    return value

alpaca_api_key_id = get_secret_or_prompt("ALPACA_API_KEY_ID")
alpaca_api_secret_key = get_secret_or_prompt("ALPACA_API_SECRET_KEY")

if not alpaca_api_key_id or not alpaca_api_secret_key:
    raise ValueError("Missing Alpaca API credentials.")

os.environ["ALPACA_API_KEY_ID"] = alpaca_api_key_id
os.environ["ALPACA_API_SECRET_KEY"] = alpaca_api_secret_key
os.environ["ALPACA_DATA_BASE_URL"] = "https://data.alpaca.markets"
os.environ["ALPACA_FEED"] = "iex"

print("ALPACA_DATA_BASE_URL:", os.environ.get("ALPACA_DATA_BASE_URL"))
print("ALPACA_FEED:", os.environ.get("ALPACA_FEED"))
print("ALPACA_API_KEY_ID and ALPACA_API_SECRET_KEY are set but not printed.")

## 4. Configure workspace, sessions, archive paths, and analysis window

These defaults keep active work under `/content` in Colab and use Google Drive only for persistence, backups, archives, and handoff packs. Replace `DRIVE_FOLDER_NAME` before any live Colab/Drive execution; the committed source intentionally guards this placeholder so it cannot create tutorial or Drive folders with a stale private path.


In [ ]:
WORKSPACE_ROOT = Path("/content") if IN_COLAB else Path.cwd()
DRIVE_FOLDER_NAME = "REPLACE_WITH_DRIVE_FOLDER_NAME"
DRIVE_ROOT = Path("/content/drive/MyDrive") / DRIVE_FOLDER_NAME if IN_COLAB else WORKSPACE_ROOT / "drive" / DRIVE_FOLDER_NAME

if DRIVE_FOLDER_NAME == "REPLACE_WITH_DRIVE_FOLDER_NAME":
    raise ValueError("Set DRIVE_FOLDER_NAME before creating Google Drive session/archive folders.")

FINTECH_ROOT = WORKSPACE_ROOT / "fintech-market-ingestion-demo"
STRATLAKE_ROOT = WORKSPACE_ROOT / "stratlake-trade-engine-demo"

FINTECH_DRIVE_ROOT = DRIVE_ROOT / "fintech-market-ingestion"
STRATLAKE_DRIVE_ROOT = DRIVE_ROOT / "stratlake-trade-engine"

FINTECH_SESSION_NAME = "fintech_stratlake_input"
STRATLAKE_SESSION_NAME = "stratlake_q1_feature_consumption"

FINTECH_SESSION_ID_OVERRIDE = ""
STRATLAKE_SESSION_ID_OVERRIDE = "stratlake_q1_feature_consumption"

ANALYSIS_START = "2026-01-02"
ANALYSIS_END = "2026-03-31"

BACKFILL_START = "2025-11-03"
BACKFILL_END = "2026-04-15"
FEATURE_BUILD_START = BACKFILL_START
FEATURE_BUILD_END = BACKFILL_END

BACKFILL_SYMBOLS = "AAPL,MSFT,NVDA,SPY,QQQ"

for path in [FINTECH_ROOT, STRATLAKE_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print("WORKSPACE_ROOT:", WORKSPACE_ROOT.as_posix())
print("DRIVE_ROOT:", DRIVE_ROOT.as_posix())
print("FINTECH_ROOT:", FINTECH_ROOT.as_posix())
print("STRATLAKE_ROOT:", STRATLAKE_ROOT.as_posix())
print("ANALYSIS window:", ANALYSIS_START, "to", ANALYSIS_END)
print("Padded backfill/build window:", BACKFILL_START, "to", BACKFILL_END)


## 5. Verify installed native CLI commands

In [ ]:
cli_commands = [
    "fintech-init-project",
    "fintech-backfill-daily",
    "fintech-backup-data",
    "stratlake-init-session",
    "stratlake-run-strategy",
    "stratlake-session-archive-bootstrap",
    "stratlake-session-archive-restore-bootstrap",
]

for cmd in cli_commands:
    result = subprocess.run([cmd, "--help"], text=True, capture_output=True)
    status = "OK" if result.returncode == 0 else f"returncode={result.returncode}"
    print(f"{cmd}: {status}")

## 6. Initialize or attach Fintech project/session

In [ ]:
fintech_init_cmd = [
    "fintech-init-project",
    "--root", FINTECH_ROOT.as_posix(),
    "--session-name", FINTECH_SESSION_NAME,
    "--with-session",
    "--colab-profile",
]

print("Fintech init command:")
print(" ".join(fintech_init_cmd))

result = subprocess.run(fintech_init_cmd, text=True, capture_output=True)
print("STDOUT:")
print(result.stdout)
if result.stderr:
    print("STDERR:")
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f"fintech-init-project failed with return code {result.returncode}")

session_manifest_candidates = sorted(
    (FINTECH_ROOT / "artifacts" / "sessions").glob("*/session_manifest.json"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

if FINTECH_SESSION_ID_OVERRIDE:
    FINTECH_SESSION_ID = FINTECH_SESSION_ID_OVERRIDE
elif session_manifest_candidates:
    FINTECH_SESSION_ID = session_manifest_candidates[0].parent.name
else:
    raise FileNotFoundError("No Fintech session manifest found after initialization.")

MARKETLAKE_ROOT = FINTECH_ROOT / "data" / "curated"
DAILY_BARS_ROOT = MARKETLAKE_ROOT / "daily_bars"

FINTECH_ARCHIVE_ID = f"curated-data-{FINTECH_SESSION_ID}"
FINTECH_DRIVE_SESSION_ROOT = FINTECH_DRIVE_ROOT / "sessions" / FINTECH_SESSION_ID
FINTECH_DRIVE_BACKUP_ROOT = FINTECH_DRIVE_SESSION_ROOT / "backups"
FINTECH_BACKUP_PACK_DIR = FINTECH_DRIVE_BACKUP_ROOT / FINTECH_ARCHIVE_ID

for path in [MARKETLAKE_ROOT, DAILY_BARS_ROOT, FINTECH_DRIVE_SESSION_ROOT, FINTECH_DRIVE_BACKUP_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print("FINTECH_SESSION_ID:", FINTECH_SESSION_ID)
print("MARKETLAKE_ROOT:", MARKETLAKE_ROOT.as_posix())
print("DAILY_BARS_ROOT:", DAILY_BARS_ROOT.as_posix())
print("FINTECH_BACKUP_PACK_DIR:", FINTECH_BACKUP_PACK_DIR.as_posix())

## 7. Initialize or attach StratLake session

In [ ]:
stratlake_init_cmd = [
    "stratlake-init-session",
    "--root", STRATLAKE_ROOT.as_posix(),
    "--project-name", STRATLAKE_SESSION_NAME,
    "--marketlake-root", MARKETLAKE_ROOT.as_posix(),
    "--drive-root", DRIVE_ROOT.as_posix(),
    "--enable-drive-persistence",
    "--notebook-configs",
]

print("StratLake init command:")
print(" ".join(stratlake_init_cmd))

result = subprocess.run(stratlake_init_cmd, text=True, capture_output=True)
print("STDOUT:")
print(result.stdout)
if result.stderr:
    print("STDERR:")
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f"stratlake-init-session failed with return code {result.returncode}")

STRATLAKE_SESSION_ID = STRATLAKE_SESSION_ID_OVERRIDE or STRATLAKE_SESSION_NAME
STRATLAKE_ARCHIVE_ID = f"stratlake-session-{STRATLAKE_SESSION_ID}"
STRATLAKE_DRIVE_SESSION_ROOT = STRATLAKE_DRIVE_ROOT / "sessions" / STRATLAKE_SESSION_ID
STRATLAKE_DRIVE_ARCHIVE_ROOT = STRATLAKE_DRIVE_SESSION_ROOT / "archives"
STRATLAKE_ARCHIVE_PACK_DIR = STRATLAKE_DRIVE_ARCHIVE_ROOT / STRATLAKE_ARCHIVE_ID

for path in [STRATLAKE_DRIVE_SESSION_ROOT, STRATLAKE_DRIVE_ARCHIVE_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

os.environ["MARKETLAKE_ROOT"] = MARKETLAKE_ROOT.as_posix()
os.environ["STRATLAKE_ROOT"] = STRATLAKE_ROOT.as_posix()

print("STRATLAKE_SESSION_ID:", STRATLAKE_SESSION_ID)
print("STRATLAKE_ARCHIVE_ID:", STRATLAKE_ARCHIVE_ID)
print("STRATLAKE_ARCHIVE_PACK_DIR:", STRATLAKE_ARCHIVE_PACK_DIR.as_posix())
print("MARKETLAKE_ROOT env:", os.environ.get("MARKETLAKE_ROOT"))
print("STRATLAKE_ROOT env:", os.environ.get("STRATLAKE_ROOT"))

## 8. Restore StratLake archive from Notebook 07/08

Archive restore is manual and off by default in committed source. Review the command preview, confirm the Notebook 07/08 archive pack exists in the configured Drive folder, and only then set `RUN_STRATLAKE_ARCHIVE_RESTORE = True` for a live runtime restore.


In [ ]:
RUN_STRATLAKE_ARCHIVE_RESTORE = False

print("STRATLAKE_ROOT:", STRATLAKE_ROOT.as_posix())
print("STRATLAKE_ARCHIVE_PACK_DIR:", STRATLAKE_ARCHIVE_PACK_DIR.as_posix())
print("Archive pack exists:", STRATLAKE_ARCHIVE_PACK_DIR.exists())

restore_cmd = [
    "stratlake-session-archive-restore-bootstrap",
    "--archive-root", STRATLAKE_ARCHIVE_PACK_DIR.as_posix(),
    "--target-root", ".",
    "--validate-before-restore",
    "--inspect-before-restore",
    "--overwrite-policy", "overwrite_allowed",
]

print("Restore command preview:")
print(" ".join(restore_cmd))

if RUN_STRATLAKE_ARCHIVE_RESTORE:
    if not STRATLAKE_ARCHIVE_PACK_DIR.exists():
        raise FileNotFoundError(
            "Expected StratLake archive pack was not found. "
            f"Run Notebook 07/08 archive checkpoint first or update STRATLAKE_SESSION_ID_OVERRIDE: {STRATLAKE_ARCHIVE_PACK_DIR.as_posix()}"
        )

    os.chdir(STRATLAKE_ROOT)
    print("Current working directory:", Path.cwd().as_posix())
    result = subprocess.run(restore_cmd, cwd=STRATLAKE_ROOT, text=True, capture_output=True)
    print("\nSTDOUT:")
    print(result.stdout)
    if result.stderr:
        print("\nSTDERR:")
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"StratLake archive restore failed with return code {result.returncode}")
else:
    print("Manual restore is off by default in committed source. Set RUN_STRATLAKE_ARCHIVE_RESTORE=True only after reviewing the command preview.")


## 9. Verify restored native StratLake inputs

In [ ]:
restored_required_paths = [
    STRATLAKE_ROOT / "configs" / "universe.yml",
    STRATLAKE_ROOT / "configs" / "paths.yml",
    STRATLAKE_ROOT / "configs" / "strategies.yml",
    STRATLAKE_ROOT / "data" / "curated" / "features_daily",
    STRATLAKE_ROOT / "artifacts",
]

print("Restored StratLake state checks:")
for p in restored_required_paths:
    print(" -", p.as_posix(), "| exists:", p.exists())

missing = [p for p in restored_required_paths[:4] if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required restored paths: " + ", ".join(p.as_posix() for p in missing))

## 10. Inspect available native strategies

In [ ]:
strategies_config_path = STRATLAKE_ROOT / "configs" / "strategies.yml"

try:
    import yaml
except Exception:
    yaml = None

strategy_names = []

if yaml is not None:
    with strategies_config_path.open("r", encoding="utf-8") as f:
        strategies_config = yaml.safe_load(f) or {}

    if isinstance(strategies_config, dict):
        if isinstance(strategies_config.get("strategies"), dict):
            strategy_names = list(strategies_config["strategies"].keys())
        elif isinstance(strategies_config.get("strategies"), list):
            for item in strategies_config["strategies"]:
                if isinstance(item, dict):
                    name = item.get("name") or item.get("strategy")
                    if name:
                        strategy_names.append(str(name))
        else:
            strategy_names = [
                str(k) for k, v in strategies_config.items()
                if isinstance(v, dict) and not str(k).startswith("_")
            ]

if not strategy_names:
    strategy_names = ["momentum_v1"]

strategy_names = sorted(dict.fromkeys(strategy_names))

print("Available native strategies:")
for name in strategy_names:
    print(" -", name)

print("strategies.yml preview:")
print(strategies_config_path.read_text(encoding="utf-8")[:4000])

## 11. Run native strategy comparison

In [ ]:
RUN_NATIVE_STRATEGY_COMPARISON = True

def extract(pattern: str, text: str, default=None, cast=None):
    match = re.search(pattern, text, flags=re.MULTILINE)
    if not match:
        return default
    value = match.group(1).strip()
    if cast is None:
        return value
    try:
        return cast(value)
    except Exception:
        return default

def extract_percent(pattern: str, text: str, default=None):
    value = extract(pattern, text, default=default, cast=float)
    if value is None:
        return default
    return value / 100.0

def parse_strategy_stdout(strategy_name: str, stdout: str, stderr: str, returncode: int) -> dict:
    return {
        "strategy": extract(r"^strategy:\s*(.+)$", stdout) or strategy_name,
        "run_id": extract(r"^run_id:\s*(.+)$", stdout),
        "analysis_start": ANALYSIS_START,
        "analysis_end": ANALYSIS_END,
        "completed": returncode == 0,
        "returncode": returncode,
        "cumulative_return": extract(r"^cumulative_return:\s*([-+0-9.]+)", stdout, cast=float),
        "sharpe_ratio": extract(r"^sharpe_ratio:\s*([-+0-9.]+)", stdout, cast=float),
        "long_pct": extract_percent(r"- long:\s*([-+0-9.]+)%", stdout),
        "short_pct": extract_percent(r"short:\s*([-+0-9.]+)%", stdout),
        "flat_pct": extract_percent(r"flat:\s*([-+0-9.]+)%", stdout),
        "trades": extract(r"- trades:\s*([0-9]+)", stdout, cast=int),
        "turnover": extract(r"turnover:\s*([-+0-9.]+)", stdout, cast=float),
        "avg_holding_bars": extract(r"- avg holding:\s*([-+0-9.]+)\s*bars", stdout, cast=float),
        "qa_status": extract(r"- status:\s*(.+)$", stdout),
        "qa_rows": extract(r"- rows:\s*([0-9]+)", stdout, cast=int),
        "qa_symbols": extract(r"symbols:\s*([0-9]+)", stdout, cast=int),
        "benchmark_return": extract_percent(r"- benchmark return:\s*([-+0-9.]+)%", stdout),
        "excess_return": extract_percent(r"- excess return:\s*([-+0-9.]+)%", stdout),
        "correlation": extract(r"- correlation:\s*([-+0-9.]+)", stdout, cast=float),
        "stderr_warning": stderr.strip() if stderr else "",
    }

strategy_rows = []
strategy_logs = {}

os.chdir(STRATLAKE_ROOT)
print("Current working directory:", Path.cwd().as_posix())

for strategy_name in strategy_names:
    cmd = [
        "stratlake-run-strategy",
        "--strategies-config", "configs/strategies.yml",
        "--strategy", strategy_name,
        "--start", ANALYSIS_START,
        "--end", ANALYSIS_END,
    ]

    print("Native strategy command:")
    print(" ".join(cmd))

    if RUN_NATIVE_STRATEGY_COMPARISON:
        result = subprocess.run(cmd, cwd=STRATLAKE_ROOT, text=True, capture_output=True)
        stdout = result.stdout or ""
        stderr = result.stderr or ""

        print("STDOUT:")
        print(stdout)
        if stderr:
            print("STDERR:")
            print(stderr)
        print("Return code:", result.returncode)

        strategy_logs[strategy_name] = {"stdout": stdout, "stderr": stderr, "returncode": result.returncode}
        strategy_rows.append(parse_strategy_stdout(strategy_name, stdout, stderr, result.returncode))
    else:
        print("Dry run only.")

strategy_comparison = pd.DataFrame(strategy_rows)

if not strategy_comparison.empty:
    sort_cols = [c for c in ["completed", "excess_return", "sharpe_ratio"] if c in strategy_comparison.columns]
    strategy_comparison = strategy_comparison.sort_values(sort_cols, ascending=[False, False, False][:len(sort_cols)])
    display(strategy_comparison)
else:
    print("No strategy comparison rows produced.")

## 12. Plot native strategy comparison

In [ ]:
if strategy_comparison.empty:
    print("No strategy_comparison dataframe available to plot.")
else:
    metric_cols = ["cumulative_return", "benchmark_return", "excess_return", "sharpe_ratio", "correlation"]
    available_metrics = [c for c in metric_cols if c in strategy_comparison.columns]

    for metric in available_metrics:
        plot_df = strategy_comparison.dropna(subset=[metric]).copy()
        if plot_df.empty:
            print(f"No values available for {metric}.")
            continue

        ax = plot_df.set_index("strategy")[metric].plot(
            kind="bar",
            title=f"Native StratLake strategy comparison: {metric}",
            figsize=(10, 4),
        )
        ax.set_xlabel("")
        ax.set_ylabel(metric)
        ax.axhline(0, linewidth=1)
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        plt.show()

    diagnostic_cols = [
        "strategy", "run_id", "qa_status", "qa_rows", "qa_symbols", "trades", "turnover",
        "avg_holding_bars", "long_pct", "short_pct", "flat_pct",
    ]
    display(strategy_comparison[[c for c in diagnostic_cols if c in strategy_comparison.columns]])

## 13. Discover native artifacts by run ID

In [ ]:
artifact_roots = [STRATLAKE_ROOT / "artifacts", STRATLAKE_ROOT / "data", STRATLAKE_ROOT / "reports"]
run_ids = [str(x) for x in strategy_comparison.get("run_id", pd.Series(dtype=str)).dropna().tolist()]
artifact_rows = []

for root in artifact_roots:
    if not root.exists():
        continue
    for p in root.rglob("*"):
        if not p.is_file():
            continue
        path_text = p.as_posix()
        matched_run_ids = [run_id for run_id in run_ids if run_id in path_text]
        if matched_run_ids or p.suffix.lower() in [".json", ".csv", ".parquet", ".md", ".html"]:
            try:
                rel = p.relative_to(STRATLAKE_ROOT).as_posix()
            except ValueError:
                rel = p.as_posix()
            artifact_rows.append({
                "matched_run_ids": ",".join(matched_run_ids),
                "relative_path": rel,
                "path": p.as_posix(),
                "suffix": p.suffix,
                "size_bytes": p.stat().st_size,
                "modified_utc": datetime.fromtimestamp(p.stat().st_mtime, tz=timezone.utc).isoformat(),
            })

artifact_inventory = (
    pd.DataFrame(artifact_rows).drop_duplicates(subset=["path"]).sort_values(["matched_run_ids", "relative_path"])
    if artifact_rows else pd.DataFrame()
)

print("Artifact rows:", len(artifact_inventory))
if artifact_inventory.empty:
    print("No native artifacts found under expected roots.")
else:
    display(artifact_inventory.head(200))

## 14. Research decision summary

In [ ]:
if strategy_comparison.empty:
    print("No strategy comparison results available.")
else:
    completed = strategy_comparison[strategy_comparison["completed"] == True].copy()
    print("Research review summary")
    print("=======================")
    print("Analysis window:", ANALYSIS_START, "to", ANALYSIS_END)
    print("Strategies attempted:", len(strategy_comparison))
    print("Strategies completed:", len(completed))

    if not completed.empty:
        if "excess_return" in completed.columns and completed["excess_return"].notna().any():
            best_excess = completed.sort_values("excess_return", ascending=False).iloc[0]
            print("Best by excess return:", best_excess["strategy"], best_excess["excess_return"])
        if "sharpe_ratio" in completed.columns and completed["sharpe_ratio"].notna().any():
            best_sharpe = completed.sort_values("sharpe_ratio", ascending=False).iloc[0]
            print("Best by Sharpe:", best_sharpe["strategy"], best_sharpe["sharpe_ratio"])
        if "turnover" in completed.columns and completed["turnover"].notna().any():
            lowest_turnover = completed.sort_values("turnover", ascending=True).iloc[0]
            print("Lowest turnover:", lowest_turnover["strategy"], lowest_turnover["turnover"])

    if "stderr_warning" in strategy_comparison.columns:
        warning_rows = strategy_comparison[strategy_comparison["stderr_warning"].fillna("").str.len() > 0]
        print("Rows with stderr warnings:", len(warning_rows))
        if len(warning_rows):
            display(warning_rows[["strategy", "run_id", "stderr_warning"]])

## 15. Optional archive checkpoint after comparison

In [ ]:
RUN_STRATLAKE_ARCHIVE_CHECKPOINT = False

archive_cmd = [
    "stratlake-session-archive-bootstrap",
    "--root", STRATLAKE_ROOT.as_posix(),
    "--archive-id", STRATLAKE_ARCHIVE_ID,
    "--archive-collision-policy", "overwrite_allowed",
    "--drive-root", STRATLAKE_DRIVE_ARCHIVE_ROOT.as_posix(),
    "--copy-policy", "overwrite_allowed",
    "--include-features",
    "--include-artifacts",
    "--include-configs",
    "--validate-after-copy",
    "--inspect-after-copy",
]

print("StratLake archive checkpoint command:")
print(" ".join(archive_cmd))

if RUN_STRATLAKE_ARCHIVE_CHECKPOINT:
    result = subprocess.run(archive_cmd, cwd=STRATLAKE_ROOT, text=True, capture_output=True)
    print("\nSTDOUT:")
    print(result.stdout)
    if result.stderr:
        print("\nSTDERR:")
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"StratLake archive checkpoint failed with return code {result.returncode}")
    print("StratLake archive checkpoint completed.")
else:
    print("Dry run only. Set RUN_STRATLAKE_ARCHIVE_CHECKPOINT=True to create/update the archive.")


## 16. Final handoff

In [ ]:
final_handoff = {
    "notebook": "Notebook 09 — Strategy Comparison and Research Review",
    "fintech_session_id": FINTECH_SESSION_ID,
    "stratlake_session_id": STRATLAKE_SESSION_ID,
    "stratlake_archive_id": STRATLAKE_ARCHIVE_ID,
    "analysis_start": ANALYSIS_START,
    "analysis_end": ANALYSIS_END,
    "strategies_attempted": int(len(strategy_comparison)) if "strategy_comparison" in globals() else 0,
    "strategies_completed": int(strategy_comparison["completed"].sum()) if "strategy_comparison" in globals() and not strategy_comparison.empty else 0,
    "artifact_rows": int(len(artifact_inventory)) if "artifact_inventory" in globals() else 0,
    "archive_pack_dir": STRATLAKE_ARCHIVE_PACK_DIR.as_posix(),
    "next_notebook": "Notebook 10 — Walk-forward / robustness / alpha or portfolio workflow",
}

display(pd.DataFrame([final_handoff]))